<a href="https://colab.research.google.com/github/241b009-max/algorithm-lab/blob/main/Lab_12.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MATRIX CHAIN MULTIPLICATION Dynamic Programming with table filling.
Problem Description:
Find minimum number of scalar multiplications needed to multiply a chain of matrices..

#Theory
Cost of multiplying two matrices:
If A is p×q and B is q×r, cost = p × q × r

Dynamic Programming idea:
Break problem into smaller subproblems.
Store results in a table to avoid recomputation.

m[i][j] = minimum cost to multiply Ai to Aj

## Algorithm
1. Take array p[] of size n
2. Create table m[n][n]
3. Set m[i][i] = 0
4. Fill table diagonally
5. For each chain length L:
    Compute m[i][j]
    Try all possible k between i and j
6. Choose minimum cost
7. Return m[1][n-1]

In [3]:
import sys

def matrix_chain_multiplication(p):
    n = len(p)

    # Create DP table for costs
    m = [[0 for _ in range(n)] for _ in range(n)]
    # Create DP table for split points (parenthesization)
    s = [[0 for _ in range(n)] for _ in range(n)]

    # L is chain length
    for L in range(2, n):
        for i in range(1, n - L + 1):
            j = i + L - 1
            m[i][j] = sys.maxsize

            for k in range(i, j):
                # cost = m[i][k] + m[k+1][j] + p[i-1]*p[k]*p[j]
                cost = m[i][k] + \
                       m[k+1][j] + \
                       p[i-1] * p[k] * p[j]

                if cost < m[i][j]:
                    m[i][j] = cost
                    s[i][j] = k  # Store the split point

    return m, s

In [2]:
# Exercise: Compute minimum cost for p = [5, 10, 3, 12, 5, 50, 6]
p_exercise = [5, 10, 3, 12, 5, 50, 6]
table_exercise = matrix_chain_multiplication(p_exercise)

n_exercise = len(p_exercise)
minimum_cost_exercise = table_exercise[1][n_exercise - 1]

print("Minimum multiplication cost for the exercise:", minimum_cost_exercise)


Minimum multiplication cost for the exercise: 2010


In [6]:
# Input dimensions
p = [10, 30, 5, 60]

table = matrix_chain_multiplication(p)

print("Minimum multiplication cost:", table[1][len(p)-1])

print("\nDP Table:")
for row in table[1:]:
    print(row[1:])

Minimum multiplication cost: [0, 0, 0, 0]

DP Table:
[[0, 0, 1, 2], [0, 0, 0, 2], [0, 0, 0, 0]]


In [4]:
def print_optimal_parenthesization(s, i, j):
    if i == j:
        return f"A{i}"
    else:
        return f"({print_optimal_parenthesization(s, i, s[i][j])}{print_optimal_parenthesization(s, s[i][j] + 1, j)})"

# Update the exercise cell to use the new function and the 's' table
p_exercise = [5, 10, 3, 12, 5, 50, 6]

m_exercise, s_exercise = matrix_chain_multiplication(p_exercise)

n_exercise = len(p_exercise)
minimum_cost_exercise = m_exercise[1][n_exercise - 1]

print("Minimum multiplication cost for the exercise:", minimum_cost_exercise)

# Print optimal parenthesization
# Note: The problem description often refers to matrices A1...An, but our 'p' array is 0-indexed.
# If 'p' has length N, it describes N-1 matrices. A1 is p[0]xp[1], A2 is p[1]xp[2], etc.
# So, the matrices are A[1] through A[n-1] in terms of typical DP problem indexing.
# For a 'p' of length 7, we have 6 matrices. So, we'd want to parenthesize matrices from index 1 to n-1 (which is 6).
optimal_paren = print_optimal_parenthesization(s_exercise, 1, n_exercise - 1)
print("Optimal Parenthesization:", optimal_paren)


Minimum multiplication cost for the exercise: 2010
Optimal Parenthesization: ((A1A2)((A3A4)(A5A6)))


In [5]:
def calculate_naive_cost(p):
    n = len(p)
    if n <= 2: # For 0 or 1 matrix, cost is 0 (or no matrices to multiply)
        return 0

    # Simulate A1 * A2 * A3 ... An-1
    # This is (A1 * A2) * A3 ...
    # Initial multiplication cost for A1*A2 (p[0]xp[1] * p[1]xp[2])
    total_cost = p[0] * p[1] * p[2] if n > 2 else 0
    current_rows = p[0] # rows of the result of A1*A2
    current_cols = p[2] # cols of the result of A1*A2

    for i in range(3, n): # A3 to An-1
        # Current result is current_rows x current_cols
        # Next matrix is p[i-1] x p[i]
        # We need to multiply (current_rows x current_cols) with (current_cols x p[i])
        total_cost += current_rows * current_cols * p[i]
        current_cols = p[i] # Update cols for the new result

    # The above logic is for left-to-right ((((A1A2)A3)A4)...)
    # A simpler way to calculate total multiplications for a linear chain A1 * A2 * ... * Ak is:
    # (p[0]*p[1]*p[2]) + (p[0]*p[2]*p[3]) + ...
    # Let's re-implement for clarity of a simple left-to-right (naive) calculation.

    # A1 is p[0]xp[1], A2 is p[1]xp[2], ..., A_k is p[k-1]xp[k]
    # We are multiplying (n-1) matrices. Let's call them M_1...M_{n-1}
    # M_1 has dimensions p[0]xp[1]
    # M_2 has dimensions p[1]xp[2]
    # ...
    # M_{n-1} has dimensions p[n-2]xp[n-1]

    # Naive (left-to-right) parenthesization: (((M1M2)M3)...M_{n-1})
    naive_cost = 0
    if n > 2: # At least two matrices (A1, A2 from p[0],p[1],p[2])
        # First product M1*M2
        naive_cost += p[0] * p[1] * p[2]
        # Resulting matrix dimensions: p[0] x p[2]

        # Subsequent products: (result) * M_i
        # For M3 (p[2]xp[3]), M4 (p[3]xp[4]), etc. up to M_{n-1} (p[n-2]xp[n-1])
        for i in range(3, n): # i represents the index in 'p' array, so p[i-1]xp[i] is the matrix
            # Dimensions of current cumulative result: p[0] x p[i-1]
            # Dimensions of next matrix M_i: p[i-1] x p[i]
            naive_cost += p[0] * p[i-1] * p[i]

    return naive_cost

naive_cost_exercise = calculate_naive_cost(p_exercise)
saved_multiplications = naive_cost_exercise - minimum_cost_exercise

print("Naive (left-to-right) multiplication cost:", naive_cost_exercise)
print("Minimum multiplication cost:", minimum_cost_exercise)
print("Multiplications saved:", saved_multiplications)


Naive (left-to-right) multiplication cost: 3380
Minimum multiplication cost: 2010
Multiplications saved: 1370


# Exercise
1. Given p = [5, 10, 3, 12, 5, 50, 6]
2. Compute minimum cost
3. Print optimal parenthesization
4. Compare number of multiplications saved